In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from scipy.sparse import csr_matrix, hstack

In [3]:
df = pd.read_csv("../data/processed/anime_cleaned.csv")

In [4]:
df["search_title"] = (
    df["title"]
    .fillna("")
    .str.lower()
)

In [5]:
metadata_columns = [
    "genres",
    "themes",
    "studios",
    "demographics"
]

for col in metadata_columns:
    df[col] = (
        df[col]
        .fillna("")
        .str.lower()
    )

df["synopsis_text"] = (
    df["synopsis"]
    .fillna("")
    .str.lower()
)

In [6]:
REMOVE_THEMES = {

    "award winning",

    "adult cast"

}

In [7]:
for col in metadata_columns:

    df[col] = df[col].apply(

        lambda x: [

            i.strip()

            for i in x.split(",")

            if i.strip() != ""

        ]

    )

In [8]:
def clean_theme_list(theme_list):

    return [

        x

        for x in theme_list

        if x not in REMOVE_THEMES

    ]

df["themes"] = df["themes"].apply(clean_theme_list)

In [9]:
df["themes"].head()

0        [space]
1        [space]
2             []
3    [detective]
4      [unknown]
Name: themes, dtype: object

In [10]:
genre_encoder = MultiLabelBinarizer()
theme_encoder = MultiLabelBinarizer()
studio_encoder = MultiLabelBinarizer()
demo_encoder = MultiLabelBinarizer()

genre_matrix = genre_encoder.fit_transform(df["genres"])
theme_matrix = theme_encoder.fit_transform(df["themes"])
studio_matrix = studio_encoder.fit_transform(df["studios"])
demo_matrix = demo_encoder.fit_transform(df["demographics"])

In [11]:
genre_sparse = csr_matrix(genre_matrix)
theme_sparse = csr_matrix(theme_matrix)
studio_sparse = csr_matrix(studio_matrix)
demo_sparse = csr_matrix(demo_matrix)

In [12]:
synopsis_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=12000,
    min_df=5
)

synopsis_matrix = synopsis_vectorizer.fit_transform(
    df["synopsis_text"]
)

In [13]:
from scipy.sparse import hstack

final_matrix = hstack([

    genre_sparse * 4,

    theme_sparse * 3,

    demo_sparse * 2,

    studio_sparse * 0.3,

    synopsis_matrix * 2

])


In [14]:
print(final_matrix.shape)

(28491, 13342)


In [15]:
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=50
)

knn_model.fit(final_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",50
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [35]:
english_alias = {

    "attack on titan": "shingeki no kyojin",
    "aot": "shingeki no kyojin",

    "demon slayer": "kimetsu no yaiba",

    "jjk": "jujutsu kaisen",

    "my hero academia": "boku no hero academia",

    "fmab": "fullmetal alchemist: brotherhood",

    "konosuba": "kono subarashii sekai ni shukufuku wo!",

    "oshi no ko": "oshi no ko",

    "spy x family": "spy x family",

    "one punch man": "one punch man",

    "tokyo ghoul": "tokyo ghoul",

    "bleach": "bleach",

    "naruto shippuden": "naruto: shippuuden",

    "dragon ball super": "dragon ball super",

    "steins gate": "steins;gate",

    "code geass": "code geass: hangyaku no lelouch"

}

In [19]:
import pandas as pd

def recommend_anime(anime_name, top_n=10):

    anime_name = anime_name.lower().strip()
    anime_name = english_alias.get(
        anime_name,
        anime_name
    )

    # -------------------------
    # Search Anime
    # -------------------------
    matches = df[
    df["search_title"].str.contains(
        anime_name.lower(),
        na=False
    )
]

    if matches.empty:
        print("❌ Anime not found.")
        return

# Prefer exact match
    exact = matches[
    matches["search_title"] == anime_name.lower()
]

    if not exact.empty:
        selected_index = exact.index[0]
    else:
        selected_index = matches.index[0]
    
    selected_title = df.loc[selected_index, "title"]

    print(f"\n🎯 Recommendations for '{selected_title}'\n")

    # -------------------------
    # KNN Search
    # -------------------------
    distances, indices = knn_model.kneighbors(
        final_matrix[selected_index],
        n_neighbors=50
    )

    recommendations = []
    used_titles = set()

    # Franchise name
    franchise = (
        selected_title
        .split(":")[0]
        .split("-")[0]
        .lower()
        .strip()
    )

    for distance, idx in zip(distances[0], indices[0]):

        if idx == selected_index:
            continue

        anime = df.iloc[idx]
        title = anime["title"].strip()

        # title = anime["title"]

        # --------------------------------
        # Remove Franchise Sequels
        # --------------------------------

        title_lower = title.lower()

        if title_lower.startswith(franchise):
            continue

        # --------------------------------
        # Skip duplicate titles
        # --------------------------------

        if title in used_titles:
            continue

        # --------------------------------
        # Skip PV / Music / Commercial
        # --------------------------------

        if anime["type"].upper() in ["PV", "CM", "MUSIC"]:
            continue

        # --------------------------------
        # Score Filter
        # --------------------------------

        score = anime["score"]

        if pd.notna(score):

            if score < 6.5:
                continue

        similarity = round((1-distance)*100,2)

        recommendations.append({

            "Title":title,

            "Similarity (%)":similarity,

            "Genres": ", ".join(anime["genres"]).title(),

            "Studio": ", ".join(anime["studios"]).title(),

            "Type":anime["type"].upper(),

            "Score":round(score,2) if pd.notna(score) else "N/A",

            "URL":anime["url"]

        })

        used_titles.add(title)

        if len(recommendations)==top_n:
            break

    result = pd.DataFrame(recommendations)

    result = result.sort_values(
        by=["Similarity (%)", "Score"],
        ascending=False
    )

    return result.reset_index(drop=True)
    # return pd.DataFrame(recommendations)

In [20]:
recommend_anime("Death Note")


🎯 Recommendations for 'Death Note'



,Title,Similarity (%),Genres,Studio,Type,Score,URL
0,Mirai Nikki: Redial,91.95,"Supernatural, Suspense",Asread.,OVA,7.28,https://myanimelist.net/anime/16762/Mirai_Nikk...
1,Shinreigari,73.15,"Mystery, Supernatural, Suspense",Production I.G,TV,7.39,https://myanimelist.net/anime/2596/Shinreigari
2,Wonder Egg Priority,72.77,"Drama, Supernatural, Suspense",Cloverworks,TV,7.55,https://myanimelist.net/anime/43299/Wonder_Egg...
3,Monogatari Series: Off & Monster Season - Zank...,67.98,"Mystery, Supernatural, Suspense",Shaft,ONA,7.73,https://myanimelist.net/anime/59612/Monogatari...
4,Mirai Nikki (TV),67.13,"Action, Supernatural, Suspense",Asread.,TV,7.38,https://myanimelist.net/anime/10620/Mirai_Nikk...
5,Jigoku Shoujo Futakomori,65.17,"Horror, Mystery, Supernatural, Suspense",Studio Deen,TV,7.87,https://myanimelist.net/anime/1594/Jigoku_Shou...
6,Kokkoku,65.15,"Drama, Mystery, Supernatural, Suspense",Geno Studio,TV,6.98,https://myanimelist.net/anime/36548/Kokkoku
7,Jigoku Shoujo,65.14,"Horror, Mystery, Supernatural, Suspense",Studio Deen,TV,7.60,https://myanimelist.net/anime/228/Jigoku_Shoujo
8,Jigoku Shoujo Mitsuganae,65.02,"Horror, Mystery, Supernatural, Suspense",Studio Deen,TV,7.60,https://myanimelist.net/anime/3713/Jigoku_Shou...
9,Cong Hong Yue Kaishi,65.02,"Horror, Sci-Fi, Supernatural, Suspense",Foch Film,ONA,7.22,https://myanimelist.net/anime/57422/Cong_Hong_...


In [21]:
recommend_anime("Shingeki no Kyojin")


🎯 Recommendations for 'Shingeki no Kyojin'



,Title,Similarity (%),Genres,Studio,Type,Score,URL
0,Jin-Rou,71.42,"Action, Award Winning, Drama, Romance, Suspense",Production I.G,MOVIE,7.78,https://myanimelist.net/anime/570/Jin-Rou
1,Sentou Yousei Yukikaze,68.51,"Action, Award Winning, Drama, Sci-Fi, Suspense",Gonzo,OVA,7.19,https://myanimelist.net/anime/1080/Sentou_Yous...
2,Evangelion Movie 1: Jo,62.79,"Action, Award Winning, Drama, Sci-Fi, Suspense",Khara,MOVIE,8.00,https://myanimelist.net/anime/2759/Evangelion_...
3,Evangelion Movie 3: Q,62.56,"Action, Award Winning, Drama, Sci-Fi, Suspense",Khara,MOVIE,7.66,https://myanimelist.net/anime/3785/Evangelion_...
4,Shin Evangelion Movie:||,62.51,"Action, Award Winning, Drama, Sci-Fi, Suspense",Khara,MOVIE,8.58,https://myanimelist.net/anime/3786/Shin_Evange...
5,Fullmetal Alchemist,62.28,"Action, Adventure, Award Winning, Drama, Fantasy",Bones,TV,8.11,https://myanimelist.net/anime/121/Fullmetal_Al...
6,Imawa no Kuni no Alice,62.21,"Action, Suspense","Silver Link., Connect",OVA,7.25,https://myanimelist.net/anime/24781/Imawa_no_K...
7,Inuyashiki,60.43,"Action, Drama, Sci-Fi, Suspense",Mappa,TV,7.63,https://myanimelist.net/anime/34542/Inuyashiki
8,Ouritsu Uchuugun: Honneamise no Tsubasa,60.43,"Action, Award Winning, Drama, Sci-Fi",Gainax,MOVIE,7.45,https://myanimelist.net/anime/1034/Ouritsu_Uch...
9,Yuugo: Koushounin,60.41,"Action, Drama, Mystery, Suspense","Artland, G&G Direction",TV,7.05,https://myanimelist.net/anime/1246/Yuugo__Kous...


In [22]:
recommend_anime("Fullmetal Alchemist")


🎯 Recommendations for 'Fullmetal Alchemist'



,Title,Similarity (%),Genres,Studio,Type,Score,URL
0,Chiisana Eiyuu: Kani to Tamago to Toumei Ningen,82.58,"Action, Adventure, Award Winning, Drama, Fantasy",Studio Ponoc,MOVIE,6.84,https://myanimelist.net/anime/37623/Chiisana_E...
1,Arslan Senki (TV): Fuujin Ranbu,82.43,"Action, Adventure, Drama, Fantasy",Lidenfilms,TV,7.50,https://myanimelist.net/anime/31821/Arslan_Sen...
2,Arslan Senki (TV),82.39,"Action, Adventure, Drama, Fantasy","Sanzigen, Lidenfilms",TV,7.65,https://myanimelist.net/anime/28249/Arslan_Sen...
3,One Piece Film: Red,80.21,"Action, Adventure, Award Winning, Comedy, Dram...",Toei Animation,MOVIE,7.82,https://myanimelist.net/anime/50410/One_Piece_...
4,Arslan Senki (TV): Tsuioku no Shou - Dakkan no...,78.02,"Action, Adventure, Drama, Fantasy","Sanzigen, Lidenfilms",TV SPECIAL,6.57,https://myanimelist.net/anime/31128/Arslan_Sen...
5,100-man no Inochi no Ue ni Ore wa Tatteiru,76.79,"Action, Adventure, Drama, Fantasy",Maho Film,TV,6.52,https://myanimelist.net/anime/41380/100-man_no...
6,Saiyuuki Gaiden,76.75,"Action, Adventure, Drama, Fantasy",Anpro,OVA,7.77,https://myanimelist.net/anime/9088/Saiyuuki_Ga...
7,Gensoumaden Saiyuuki,76.70,"Action, Adventure, Drama, Fantasy",Pierrot,TV,7.54,https://myanimelist.net/anime/129/Gensoumaden_...
8,100-man no Inochi no Ue ni Ore wa Tatteiru 2nd...,76.67,"Action, Adventure, Drama, Fantasy",Maho Film,TV,6.79,https://myanimelist.net/anime/44881/100-man_no...
9,One Piece Movie 06: Omatsuri Danshaku to Himit...,76.66,"Action, Adventure, Drama, Fantasy",Toei Animation,MOVIE,7.78,https://myanimelist.net/anime/464/One_Piece_Mo...


In [34]:
recommend_anime("Kimetsu no Yaiba")


🎯 Recommendations for 'Kimetsu no Yaiba'



,Title,Similarity (%),Genres,Studio,Type,Score,URL
0,Jujutsu Kaisen,80.04,"Action, Award Winning, Supernatural",Mappa,TV,8.55,https://myanimelist.net/anime/40748/Jujutsu_Ka...
1,Donten ni Warau,72.66,"Action, Supernatural",Doga Kobo,TV,7.46,https://myanimelist.net/anime/21743/Donten_ni_...
2,"Donten ni Warau Gaiden: Ketsubetsu, Yamainu no...",72.53,"Action, Supernatural",Wit Studio,MOVIE,7.40,https://myanimelist.net/anime/35086/Donten_ni_...
3,"Donten ni Warau Gaiden: Ouka, Tenbou no Kakehashi",72.53,"Action, Supernatural",Wit Studio,MOVIE,7.21,https://myanimelist.net/anime/35425/Donten_ni_...
4,Sirius,66.94,"Action, Supernatural",P.A. Works,TV,6.95,https://myanimelist.net/anime/37569/Sirius
5,Shounen Onmyouji,66.87,"Action, Supernatural",Studio Deen,TV,7.53,https://myanimelist.net/anime/1557/Shounen_Onm...
6,"Donten ni Warau Gaiden: Shukumei, Soutou no Fuuma",66.68,"Action, Supernatural",Wit Studio,MOVIE,7.51,https://myanimelist.net/anime/35424/Donten_ni_...
7,Nurarihyon no Mago: Sennen Makyou Recaps,66.45,"Action, Supernatural",Studio Deen,TV SPECIAL,6.68,https://myanimelist.net/anime/11083/Nurarihyon...
8,Chrno Crusade,65.18,"Action, Romance, Supernatural",Gonzo,TV,7.60,https://myanimelist.net/anime/60/Chrno_Crusade
9,Kuroshitsuji II,64.97,"Action, Comedy, Supernatural",A-1 Pictures,TV,7.12,https://myanimelist.net/anime/6707/Kuroshitsuj...


In [25]:
recommend_anime("Code Geass")


🎯 Recommendations for 'Code Geass: Hangyaku no Lelouch'



,Title,Similarity (%),Genres,Studio,Type,Score,URL
0,Shin Kidou Senki Gundam Wing: Endless Waltz,83.25,"Action, Award Winning, Drama, Sci-Fi",Sunrise,OVA,7.73,https://myanimelist.net/anime/91/Shin_Kidou_Se...
1,Soukyuu no Fafner: Dead Aggressor - The Beyond,80.37,"Action, Drama, Sci-Fi","Production I.G, I.Gzwei",MOVIE,7.27,https://myanimelist.net/anime/34649/Soukyuu_no...
2,Guilty Crown,78.62,"Action, Drama, Sci-Fi",Production I.G,TV,7.40,https://myanimelist.net/anime/10793/Guilty_Crown
3,Taiyou no Kiba Dougram,78.47,"Action, Drama, Sci-Fi",Sunrise,TV,7.40,https://myanimelist.net/anime/2257/Taiyou_no_K...
4,86 Part 2,78.44,"Action, Drama, Sci-Fi",A-1 Pictures,TV,8.72,https://myanimelist.net/anime/48569/86_Part_2
5,86,78.39,"Action, Drama, Sci-Fi",A-1 Pictures,TV,8.33,https://myanimelist.net/anime/41457/86
6,Soukou Kihei Votoms: Big Battle,78.39,"Action, Drama, Sci-Fi",Sunrise,OVA,6.71,https://myanimelist.net/anime/2584/Soukou_Kihe...
7,Kidou Senshi Gundam 00 Special Edition,78.32,"Action, Drama, Sci-Fi",Sunrise,OVA,7.53,https://myanimelist.net/anime/7270/Kidou_Sensh...
8,Soukou Kihei Votoms: The Last Red Shoulder,78.32,"Action, Drama, Sci-Fi",Sunrise,OVA,7.11,https://myanimelist.net/anime/2583/Soukou_Kihe...
9,Kidou Senshi Gundam AGE: Memory of Eden,78.32,"Action, Drama, Sci-Fi",Sunrise,OVA,6.62,https://myanimelist.net/anime/17655/Kidou_Sens...


In [27]:
recommend_anime("Naruto")


🎯 Recommendations for 'Naruto'



,Title,Similarity (%),Genres,Studio,Type,Score,URL
0,Boruto: Naruto Next Generations Part 2,96.74,"Action, Adventure, Fantasy",Unknown,TV,N/A,https://myanimelist.net/anime/54687/Boruto__Na...
1,Boruto: Naruto the Movie,95.84,"Action, Adventure, Fantasy",Pierrot,MOVIE,7.38,https://myanimelist.net/anime/28755/Boruto__Na...
2,Dragon Ball Z Movie 14: Kami to Kami,93.79,"Action, Adventure, Fantasy",Toei Animation,MOVIE,7.41,https://myanimelist.net/anime/14837/Dragon_Bal...
3,Dragon Ball Super: Super Hero,93.77,"Action, Adventure, Fantasy",Toei Animation,MOVIE,7.58,https://myanimelist.net/anime/48903/Dragon_Bal...
4,Dragon Ball Super: Broly,93.75,"Action, Adventure, Fantasy",Toei Animation,MOVIE,8.19,https://myanimelist.net/anime/36946/Dragon_Bal...
5,Juushin Enbu: Hero Tales,93.72,"Action, Adventure, Fantasy",Studio Flag,TV,6.81,https://myanimelist.net/anime/2772/Juushin_Enb...
6,Ba Jie: Tian Peng Xiajie,90.39,"Action, Adventure, Fantasy",Unknown,MOVIE,N/A,https://myanimelist.net/anime/57931/Ba_Jie__Ti...
7,Son Ogong,90.39,"Action, Adventure, Fantasy",Unknown,MOVIE,N/A,https://myanimelist.net/anime/42292/Son_Ogong
8,Xian Ji Xun Zong,90.39,"Action, Adventure, Fantasy",Ruo Hong Culture,ONA,N/A,https://myanimelist.net/anime/60066/Xian_Ji_Xu...
9,Xin Xiyou Lixian Ji,90.39,"Action, Adventure, Fantasy",Unknown,TV,N/A,https://myanimelist.net/anime/58557/Xin_Xiyou_...


In [28]:
import joblib
from scipy.sparse import save_npz

# Save KNN model
joblib.dump(knn_model, "knn_model.pkl")

# Save feature matrix
save_npz("final_matrix.npz", final_matrix)

# Save processed dataset
df.to_csv("anime.csv", index=False)

print("✅ Everything saved successfully!")

✅ Everything saved successfully!


In [32]:
print(df["search_title"].head(10))

0                       cowboy bebop
1    cowboy bebop: tengoku no tobira
2                             trigun
3                 witch hunter robin
4                     bouken ou beet
5                       eyeshield 21
6               hachimitsu to clover
7         hungry heart: wild striker
8             initial d fourth stage
9                            monster
Name: search_title, dtype: str
